#training instute analytics & Revenue Intelligence

#Step 1 : connect python tp mysql

In [4]:
import mysql.connector
import pandas as pd
import random
from datetime import date,timedelta

conn= mysql.connector.connect(
    host="localhost",
    user="root",
    password="root"
)
cursor=conn.cursor()

#step2:create database

In [8]:
cursor.execute("create database if not exists institute_analytics")
cursor.execute("use institute_analytics")

#create tables

In [14]:
cursor.execute("""
create table if not exists students(
student_id int auto_increment primary key,
student_name varchar(100),
city varchar(25),
join_date date
)
""")
cursor.execute("""
create table if not exists courses(
course_id int auto_increment primary key,
course_name varchar(100),
course_fees int
)
""")
cursor.execute("""
create table if not exists enrollments(
    enrollment_id int auto_increment primary key,
    student_id int,
    course_id int,
    enrollment_date date,
    fees_paid int,
    foreign key (student_id) references students(student_id),
    foreign key (course_id) references courses(course_id)
)
""")


In [17]:
cursor.execute("""
create table if not exists student_mark(
    mark_id int auto_increment primary key,
    student_id int,
    subject varchar(50),
    mark int,
    exam_date date,
    foreign key (student_id) references students(student_id)
)
""")
conn.commit()

#insert courses

In [19]:
courses=[
    ("python",300000),
    ("data analytics",400000),
    ("machine learning",500000),
    ("power BI",250000)
]
cursor.executemany(
    "insert into courses (course_name,course_fees) values (%s,%s)",
    courses
)
conn.commit()


#insert students

In [24]:
cities=["pune","mumbai","nashik","nagpur"]
students=[]

for i in range(20):
    students.append((
        f"student_{i+1}",
        random.choice(cities),
        date(2024,1,1)+ timedelta(days=random.randint(1,300))
    ))

cursor.executemany(
    "insert into students(student_name,city,join_date) values (%s,%s,%s)",
    students
)
conn.commit()

#insert

In [33]:
enrollments=[]
for _ in range(40):
    enrollments.append((
        random.randint(1,20),
        random.randint(1,4),
        date(2024,1,1) + timedelta(days=random.randint(1,300)),
        random.randint(20000,50000)
    ))

cursor.executemany("""
insert into enrollments
(student_id,course_id,enrollment_date,fees_paid)
values (%s,%s,%s,%s)
""",enrollments)
conn.commit()

#insert marks

In [54]:

subjects = ["python", "java", "mysql", "ML"]
marks_data = []

for _ in range(100):
    marks_data.append((
        random.randint(1, 20),
        random.choice(subjects), 
        random.randint(30, 100),
        date(2024, 6, 1)
    ))

cursor.executemany("""
insert into student_mark
(student_id, subject, mark, exam_date)
values (%s, %s, %s, %s)
""", marks_data)

conn.commit()

#1)average marks per students

In [55]:
query="""
select student_id, avg(mark) as avg_marks
from student_mark
group by student_id"""

performance_df = pd.read_sql(query,conn)
performance_df.head()

C:\Users\akash\AppData\Local\Temp\ipykernel_9084\674232264.py:6: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  performance_df = pd.read_sql(query,conn)


,student_id,avg_marks
0,1,69.7778
1,2,55.6667
2,3,52.0000
3,4,60.3333
4,5,59.6000


#Add performance

In [ ]:
def classify(marks):
    if marks>75:
        return"Distincton"
    elif marks>=60:
        return "First class"
    elif marks>=40:
        return "second Class"
    else:
        return "fail"

performance_df["grand"